# Notebook 03 — Precision Coding

**Duration:** ~30 minutes

In notebook 02 you saw that prompt structure matters. In this notebook we take that further. You will:

1. Build the same request five ways, adding one layer of prompt structure at a time, and see exactly how the output changes.
2. Replicate a published coding method (Tai et al., 2024) by coding a passage from the Privacy Act 2020 for the presence or absence of specific concepts — first by hand, then with the LLM.
3. When you and the LLM disagree, ask the LLM to defend itself, and decide whether its reasoning changes your mind.

The published paper behind this workshop (Ardekani et al., 2026) reports LLM extraction at 86% precision against 96% for rule-based methods. Prompt design is one of the main reasons for that gap. This notebook makes that gap concrete.

## Setup

Run this cell first. It loads your base URL and API key from the `.env` file you set up in notebook 01.

In [19]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from openai import OpenAI, AuthenticationError, APIConnectionError
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key.
# In VS Code they come from the .env file (set up in notebook 01).
# In Google Colab there is no .env file, so the cell asks you to paste them in.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from getpass import getpass
    os.environ["LLM_BASE_URL"] = input("Paste your base URL: ").strip()
    os.environ["LLM_API_KEY"] = getpass("Paste your API key: ").strip()
else:
    from dotenv import load_dotenv
    load_dotenv(override=True)

if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    if IN_COLAB:
        raise RuntimeError("Base URL or API key is empty. Run this cell again and paste both values.")
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Ask the machine which model it serves. Each machine serves one model, and
# not every machine serves the same one, so we look the name up.
# The model reads both text and images.
try:
    MODEL = client.models.list().data[0].id
except AuthenticationError:
    raise RuntimeError("The machine rejected your API key. Check it, then run this cell again.") from None
except APIConnectionError:
    raise RuntimeError(
        "Could not reach the machine. Check your base URL. "
        "In VS Code, also check that you are on the university VPN."
    ) from None

# To choose a model yourself, remove the # at the start of the next line
# and put the model's name between the quote marks.
# MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

Using model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4.
Setup complete.


## The passage we will work with

Today we use a short passage from New Zealand's **Privacy Act 2020**. This Act sets out the **Information Privacy Principles** (IPPs) — 13 rules that agencies must follow when they handle personal information about people.

We will focus on two of those principles:
- **IPP 3** — Collection of information from the person it is about.
- **IPP 11** — Limits on disclosing personal information.

These are substantive rules, written in plain-ish legal English. They give us enough material to summarise, code, and debate.

> Note: the passage below has been adapted for teaching. It follows the structure and wording of the Act but may not be word-for-word with the latest published text. Use the Act itself as the authoritative source for any research work.

Run the cell below to load the passage.

In [20]:
PRIVACY_ACT_PASSAGE = """
Information Privacy Principle 3 — Collection of information from subject.

Where an agency collects personal information directly from the individual concerned, the agency must take any steps that are, in the circumstances, reasonable to ensure that the individual concerned is aware of:
(a) the fact that the information is being collected;
(b) the purpose for which the information is being collected;
(c) the intended recipients of the information;
(d) the name and address of the agency that is collecting the information and the agency that will hold the information;
(e) whether the supply of the information is voluntary or mandatory, and, if mandatory, the law under which it is required;
(f) the consequences (if any) for the individual if the information is not provided;
(g) the rights of access to, and correction of, that personal information.

Information Privacy Principle 11 — Limits on disclosure of personal information.

An agency that holds personal information must not disclose the information to any other agency or to any person unless the agency believes, on reasonable grounds, that:
(a) the disclosure is one of the purposes in connection with which the information was obtained, or is directly related to those purposes;
(b) the source of the information is a publicly available publication;
(c) the disclosure is authorised by the individual concerned;
(d) non-compliance is necessary to prevent or lessen a serious threat to public health or safety, or to the life or health of any individual.
"""

print(PRIVACY_ACT_PASSAGE)
print(f"Passage length: {len(PRIVACY_ACT_PASSAGE.split())} words")


Information Privacy Principle 3 — Collection of information from subject.

Where an agency collects personal information directly from the individual concerned, the agency must take any steps that are, in the circumstances, reasonable to ensure that the individual concerned is aware of:
(a) the fact that the information is being collected;
(b) the purpose for which the information is being collected;
(c) the intended recipients of the information;
(d) the name and address of the agency that is collecting the information and the agency that will hold the information;
(e) whether the supply of the information is voluntary or mandatory, and, if mandatory, the law under which it is required;
(f) the consequences (if any) for the individual if the information is not provided;
(g) the rights of access to, and correction of, that personal information.

Information Privacy Principle 11 — Limits on disclosure of personal information.

An agency that holds personal information must not disclose

## Act 1 — Exploratory analysis progression

We will summarise the passage above five times. Each version adds one more layer of prompt structure. After all five, we compare them side by side with automatic word counts.

A word on terminology:
- **Prompt** — the text you send to the LLM.
- **Role** — a line that tells the model who it should pretend to be. Sent as the first message of the conversation.
- **Output format** — rules about how the answer should look (length, style, structure).
- **Perspective** — a stance or viewpoint the model should take (e.g. a privacy rights advocate, a business owner).

We will keep `temperature=0` throughout, so any differences come from the prompt, not randomness.

Each answer is saved into a dictionary called `outputs` so we can compare them at the end.

### Version 1 — bare instruction

**Your task:** Write the simplest prompt you can think of — one that asks the model to summarise the passage. No role, no word limit, no formatting rules. Just the bare request.

In the cell below, fill in the `content` field and include the passage inside your prompt by writing `{PRIVACY_ACT_PASSAGE}` like this:

```
{"role": "user", "content": "Provide a detailed explanation of the following passage: {PRIVACY_ACT_PASSAGE}"}
```



In [ ]:
outputs = {}

response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "user", "content": f""}   # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v1 — bare instruction"] = response.choices[0].message.content.strip()
print(outputs["v1 — bare instruction"])

**Summary**

- **Collection of personal information (Principle 3):**  
  When an agency gathers data directly from an individual, it must, in a reasonable manner, ensure the person knows:  
  1. The information is being collected.  
  2. Why it is being collected.  
  3. Who will receive the information.  
  4. The collecting agency and the agency that will retain it.  
  5. Whether providing the information is optional or required by law (and the legal basis if required).  
  6. Any consequences for the individual if they do not provide it.  
  7. Their rights to access and correct the information.

- **Limits on disclosure (Principle 11):**  
  An agency may not share personal information with another agency or person unless it reasonably believes one of the following applies:  
  1. The disclosure is linked to the original purpose(s) for which the data were collected.  
  2. The source of the data is publicly available.  
  3. The individual has given consent.  
  4. Disclosure is n

### Version 2 — add a word limit

**Your task:** Write a prompt that asks for a summary of the passage **in under 20 words**. 

You might have to try this a few times.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "user", "content": f""}   # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v2 — + word limit"] = response.choices[0].message.content.strip()
print(outputs["v2 — + word limit"])

### Version 3 — add a role

**Your task:** Keep the word limit from v2, and now add a **role** by writing a system message. Roles narrow the model's focus by pointing it at a specific professional frame.

Have a go at describing yourself. E.g., "I'm a PhD reseaerch with a background in privacy law from New Zealand and I specialise in dog attacks, and I surf on the weekends."

Fill in both `content` fields below — the system message (the role) and the user message (the same request as v2).

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "system", "content": ""},   # <-- WRITE YOUR ROLE HERE
        {"role": "user", "content": f""}     # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v3 — + role"] = response.choices[0].message.content.strip()
print(outputs["v3 — + role"])

### Version 4 — add output format rules

**Your task:** Keep the role and the word limit from v3, and now add **output format rules** to the user message. Ask the model to:

- use **plain English**,
- **avoid legal jargon**,
- return **only the summary** with **no preamble** (so it does not start with phrases like "Here is a summary:").

Fill in both `content` fields below.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "system", "content": ""},   # <-- WRITE THE ROLE HERE
        {"role": "user", "content": f""}     # <-- WRITE YOUR PROMPT HERE
    ]
)
outputs["v4 — + output format"] = response.choices[0].message.content.strip()
print(outputs["v4 — + output format"])

### Version 5 — add a perspective (your turn)

This is a fill-in-the-blanks cell. The role now carries a **perspective** — a stance the model should take. The formatting rules are exactly the same as v4.

We suggest starting with: **"a privacy rights advocate concerned with surveillance risks"**. Once you have run it, try other perspectives — a small business owner, a data scientist, a teenager — and see what changes.

In [ ]:
# Fill in the blank below with a perspective. Suggested starting value:
#   "a privacy rights advocate concerned with surveillance risks"
perspective = "a privacy rights advocate concerned with surveillance risks"

v5_system = f"You are {perspective}."
v5_user = (
    "Summarise this section of the Privacy Act in under 20 words. "
    "Use plain English. Avoid legal jargon. Return only the summary. No preamble.\n\n"
    f"{PRIVACY_ACT_PASSAGE}"
)

response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "system", "content": v5_system},
        {"role": "user", "content": v5_user}
    ]
)
outputs["v5 — + perspective"] = response.choices[0].message.content.strip()
print(outputs["v5 — + perspective"])

### Compare the five versions

The cell below prints a table with the version label, automatic word count, and a trimmed view of each output, followed by the full outputs in order.

In [ ]:
print(f"{'Version':<28} {'Words':>6}   Output (first 60 chars)")
print("-" * 100)
for label, text in outputs.items():
    word_count = len(text.split())
    first_line = text.replace("\n", " ")
    if len(first_line) > 60:
        first_line = first_line[:57] + "..."
    print(f"{label:<28} {word_count:>6}   {first_line}")

print()
print("Full outputs:")
print("=" * 60)
for label, text in outputs.items():
    print(f"\n[{label}]  ({len(text.split())} words)")
    print(text)

### Discussion

Look at the table above and answer these for yourself:

1. **Which version would you cite in a methods section?** 
2. **Which version would give the most consistent results if you ran it again?**
3. **Why does v5 emphasise different things from v4, even though the format rules are identical?** 

You can try this lever again in notebook 04, where an optional exercise asks the model to describe an image from a different perspective.

## Act 2 — Presence / absence coding (Tai et al. replication)

> Tai et al. (2024) used an LLM to code whether specific psychological constructs — words like persistence, admiration, frustration — were present or absent in interview transcripts. They found the LLM agreed with human coders at a rate comparable to inter-rater reliability between two humans. We are going to replicate that logic using the Privacy Act 2020.

**Presence/absence coding** is one of the simplest forms of content coding. For each concept in your coding scheme, you mark `1` if it is present in the text (directly stated or clearly implied) and `0` if it is absent. It is a standard technique in qualitative and mixed-methods research.

We will code the passage above for these eight concepts:

- **consent** — did the individual agree to what is happening to their information?
- **individual** — is a specific person the subject of the rule?
- **purpose** — why is the information being collected or disclosed?
- **disclosure** — is information being shared with others?
- **collect** — is information being gathered?
- **payment** — is money being paid or charged?
- **children** — are children or young people specifically mentioned?
- **employment** — is the information about someone's job or workplace?

You will do this **twice**: first by yourself, then with the LLM. Then we compare.

### Step 1 — Your coding

Read the passage one more time (it is printed below as a reminder). For each keyword, decide: is the concept **present** (directly stated or clearly implied) or **absent**?

In [22]:
print(PRIVACY_ACT_PASSAGE)


Information Privacy Principle 3 — Collection of information from subject.

Where an agency collects personal information directly from the individual concerned, the agency must take any steps that are, in the circumstances, reasonable to ensure that the individual concerned is aware of:
(a) the fact that the information is being collected;
(b) the purpose for which the information is being collected;
(c) the intended recipients of the information;
(d) the name and address of the agency that is collecting the information and the agency that will hold the information;
(e) whether the supply of the information is voluntary or mandatory, and, if mandatory, the law under which it is required;
(f) the consequences (if any) for the individual if the information is not provided;
(g) the rights of access to, and correction of, that personal information.

Information Privacy Principle 11 — Limits on disclosure of personal information.

An agency that holds personal information must not disclose

Edit the `your_coding` dictionary in the cell below — replace each `None` with `1` for present or `0` for absent. **Do this before running the LLM cell** so you are not biased by the model's answer.

In [ ]:
print("=" * 60)

KEYWORDS = ["consent", "individual", "purpose", "disclosure", "collect", "payment", "children", "employment"]

# Read the passage above. For each keyword, enter 1 if the concept is present
# (directly stated or clearly implied) or 0 if absent. Do this BEFORE running
# the next cell.
your_coding = {
    "consent": None, # put 1 or 0 here
    "individual": None,
    "purpose": None,
    "disclosure": None,
    "collect": None,
    "payment": None,
    "children": None,
    "employment": None,
}

# Sanity check — reminds you if any values are still None
if any(v is None for v in your_coding.values()):
    print("Warning: some keywords still have value None. Fill them in before running the next cell.")
else:
    print("Your coding is complete:")
    for k, v in your_coding.items():
        print(f"  {k:<12} -> {v}")

Your coding is complete:
  consent      -> 1
  individual   -> 1
  purpose      -> 1
  disclosure   -> 0
  collect      -> 0
  payment      -> 1
  children     -> 1
  employment   -> 0


### Step 2 — LLM coding

Now we ask the LLM the same question. We use a strict prompt: return only a JSON object, nothing else. That makes the response easy to parse programmatically.

JSON stands for **JavaScript Object Notation**. It is a simple text format for structured data — a set of key–value pairs in curly braces, like `{"consent": 1, "individual": 0}`.

In [24]:
coding_system = (
    "You are a qualitative researcher performing presence/absence coding on legislative text. "
    "For each keyword, decide whether the concept is present (directly stated or clearly implied) or absent. "
    "Return ONLY a JSON object mapping each keyword to 1 (present) or 0 (absent). "
    "No explanation. No preamble. No code fences."
)

coding_user = (
    f"Text:\n{PRIVACY_ACT_PASSAGE}\n\n"
    f"Keywords to code: {KEYWORDS}\n\n"
    "Return a JSON object with each keyword as a key and 1 or 0 as the value."
)

response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "system", "content": coding_system},
        {"role": "user", "content": coding_user}
    ]
)

raw_reply = response.choices[0].message.content.strip()
print("LLM raw reply:")
print(raw_reply)
print()

# Parse JSON. Strip markdown code fences if the model added them.
def parse_json_reply(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        lines = [l for l in lines if not l.startswith("```")]
        text = "\n".join(lines).strip()
    return json.loads(text)

try:
    parsed = parse_json_reply(raw_reply)
    # Ensure all eight keywords are present; anything missing defaults to 0
    llm_coding = {k: int(parsed.get(k, 0)) for k in KEYWORDS}
    print("Parsed LLM coding:")
    for k, v in llm_coding.items():
        print(f"  {k:<12} -> {v}")
except (json.JSONDecodeError, ValueError, TypeError) as err:
    print(f"Could not parse LLM output as JSON. Error: {err}")
    print("Falling back to zeros. Re-run the cell or inspect the raw reply above.")
    llm_coding = {k: 0 for k in KEYWORDS}

LLM raw reply:
{"consent": 1, "individual": 1, "purpose": 1, "disclosure": 1, "collect": 1, "payment": 0, "children": 0, "employment": 0}

Parsed LLM coding:
  consent      -> 1
  individual   -> 1
  purpose      -> 1
  disclosure   -> 1
  collect      -> 1
  payment      -> 0
  children     -> 0
  employment   -> 0


### Step 3 — Comparison

Put the two codings side by side and count the agreements.

In [25]:
print(f"{'Keyword':<12} {'Your coding':<14} {'LLM coding':<14} Agreement")
print("-" * 55)

agreements = 0
for k in KEYWORDS:
    your_val = your_coding[k]
    llm_val = llm_coding.get(k)
    agree = your_val == llm_val
    if agree:
        agreements += 1
    mark = "agree" if agree else "DISAGREE"
    your_display = str(your_val) if your_val is not None else "(not coded)"
    print(f"{k:<12} {your_display:<14} {str(llm_val):<14} {mark}")

print()
print(f"You agreed with the LLM on {agreements} out of {len(KEYWORDS)} keywords.")

Keyword      Your coding    LLM coding     Agreement
-------------------------------------------------------
consent      1              1              agree
individual   1              1              agree
purpose      1              1              agree
disclosure   0              1              DISAGREE
collect      0              1              DISAGREE
payment      1              0              DISAGREE
children     1              0              DISAGREE
employment   0              0              agree

You agreed with the LLM on 4 out of 8 keywords.


## Act 3 — Disagreement resolution

When you and the LLM disagree, one of you is wrong — or you are reading the text through different lenses. Either way, the disagreement is useful information. This is the punchline of the whole exercise: **disagreement is not failure, it is evidence about how well the method is working**.

We will now find every disagreement and ask the LLM to explain its coding.

An LLM has no memory between calls. Each request starts from a blank slate. To ask a follow-up question, we send the whole conversation again: the original instructions, our coding request, and the LLM's own JSON reply. Then we add the new question at the end. This is how every chat tool works behind the scenes.

One caution. The LLM gave its codes without explaining them. Any explanation it gives now is written after the fact, to justify an answer it already gave. Treat it as a claim to check against the text, not as a record of how the model decided.

In [ ]:
disagreements = [k for k in KEYWORDS if your_coding[k] != llm_coding[k]]

if not disagreements:
    print("No disagreements. To try this step anyway, flip one of your values")
    print("in the `your_coding` cell, run that cell again, then run this cell.")

# Rebuild the conversation from Step 2: instructions, our request, the LLM's reply.
history = [
    {"role": "system", "content": coding_system},
    {"role": "user", "content": coding_user},
    {"role": "assistant", "content": raw_reply},
]

for k in disagreements:
    question = (
        f"You coded '{k}' as {llm_coding[k]}. Explain why in 2-3 plain sentences "
        "(not JSON). Quote the part of the text that led to your decision."
    )
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=0,
        messages=history + [{"role": "user", "content": question}],
    )
    print("=" * 60)
    print(f"{k}: you coded {your_coding[k]}, the LLM coded {llm_coding[k]}")
    print(response.choices[0].message.content.strip())
    print()

### Reflection

When you disagreed with the LLM and asked for its reasoning — did the explanation change your mind? If yes, the LLM surfaced something you missed. If no, the LLM made an error your manual check caught. Both outcomes are valid data about the reliability of this method. Tai et al. (2024) found LLMs agreed with human coders approximately 86% of the time on presence/absence coding tasks. How does your agreement rate today compare? What would you do differently if you were coding 500 sections rather than one?

## Test yourself

Two short questions to check what you have learned in this notebook. There is no score and nobody is marking you. Talk your answers through with the person next to you if you like.


### Question 1 — Fix a weak prompt

A colleague sent the prompt below. They got back a 60-word answer that started with "Sure! Here's a summary of the text:".

```
{"role": "user", "content": f"Tell me about this. Keep it short. {PRIVACY_ACT_PASSAGE}"}
```

**a)** Which of the four layers from Act 1 are missing? (Word limit, role, output format, perspective.)

**b)** Rewrite the prompt in the cell below so that the answer:

- is **under 25 words**,
- uses **plain English** with **no preamble**,
- is written for **a volunteer at a community law centre**.

Run the cell. It prints the word count so you can check that you met the limit.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {"role": "system", "content": ""},   # <-- WRITE YOUR ROLE / PERSPECTIVE HERE
        {"role": "user", "content": f""}     # <-- WRITE YOUR PROMPT HERE (include {PRIVACY_ACT_PASSAGE})
    ]
)
answer = response.choices[0].message.content.strip()
print(answer)
print()
print(f"Word count: {len(answer.split())}")


### Question 2 — A convincing explanation

Imagine you coded **consent** as `0`, and the LLM coded it as `1`. You asked the LLM to explain itself, and it said:

> "The passage states that information may only be disclosed **with the express written consent of the individual**, which directly establishes consent."

What should you do next?

- **A)** Change your code to `1`. The LLM's explanation is specific and confident.
- **B)** Keep your `0`. LLMs are not reliable coders.
- **C)** Look for the quoted phrase in the passage. If it is not there, record the explanation as a **hallucination** (something the model made up). Then decide on consent using the passage's actual wording.
- **D)** Run the prompt again until the LLM agrees with you.

Pick your answer, then use the cell below to check the LLM's quote against the passage. Does the result change your answer?


In [ ]:
# Does the LLM's quoted phrase actually appear in the passage?
quote = "express written consent"
print(f"'{quote}' in passage: {quote in PRIVACY_ACT_PASSAGE}")

# Now try a phrase of your own from the passage that relates to consent.


## What you accomplished

In this notebook you:

- Saw five versions of the same prompt and measured exactly how each layer of structure (word limit, role, output format, perspective) changes the output.
- Replicated the presence/absence coding method from Tai et al. (2024) on a passage of real NZ legislation.
- Compared your own coding to the LLM's, counted agreements, and asked the LLM to defend each disagreement.

The key insight: a prompt is not a casual question, it is a **method**. The clearer and more specific you make it, the more reproducible your results become — and the more honestly you can describe what you did in a methods section.

**Next up:** In this notebook you controlled the model's output through prompt structure, and you tested its coding reliability against your own judgement. In the next notebook we use those same skills — structured prompts and checking the model against a trusted source — on a new kind of data: images.